# Stage 1A official `circuit-tracer` reproduction

This notebook is a thin Colab/CUDA orchestrator for tracked Stage 1A scripts. It contains no scientific implementation and has not been executed. Before running it, select a Colab runtime with **Python 3.11** and a GPU; the planned target is Colab runtime `2025.07`. The notebook applies an **unverified preflight policy** of at least 14 GiB total and 12 GiB free VRAM before retaining the official `batch_size=256`. This is not a measured requirement or completion guarantee: an OOM may still occur and the tracked runner will preserve a sanitized diagnostic. It fails closed on a different Python version, the stated VRAM policy, missing CUDA/BF16 support, missing immutable pins, or invalid artifacts.

Before execution, paste the final 40-character `stage-1a-reproduction` commit into `EXPECTED_PROJECT_COMMIT` in the first code cell. The notebook refuses to run a different branch head. Add a Colab secret named `HF_TOKEN` with access to `google/gemma-2-2b`. The token is passed only through the child-process environment and is never displayed or serialized. Raw model, transcoder, graph, and cache artifacts remain outside the Git bundle.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shlex
import subprocess
import sys
import zipfile
from pathlib import Path

REPOSITORY = "https://github.com/eokahya/counterfactual-susceptibility.git"
PROJECT_REF = "stage-1a-reproduction"
EXPECTED_PROJECT_COMMIT = ""  # @param {type:"string"}
UPSTREAM_COMMIT = "8f1e2438df612464e229e44c4a00ff637bf9379b"
MODEL_ID = "google/gemma-2-2b"
MODEL_REVISION = "c5ebcd40d208330abc697524c919956e692655cf"
TRANSCODER_ID = "mwhanna/gemma-scope-transcoders"
TRANSCODER_REVISION = "bd5773156dea09893636c801df1237d0410307d2"
REPOSITORY_DIR = Path("/content/counterfactual-susceptibility")
VENV_DIR = Path("/content/cfsus-stage1a-venv")

for revision in (
    EXPECTED_PROJECT_COMMIT,
    UPSTREAM_COMMIT,
    MODEL_REVISION,
    TRANSCODER_REVISION,
):
    if re.fullmatch(r"[0-9a-f]{40}", revision) is None:
        raise ValueError(f"Expected an immutable 40-character SHA, got {revision!r}")


def run(
    argv: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> None:
    print("+", shlex.join(argv))
    subprocess.run(argv, cwd=cwd, env=env, check=True)


def capture(argv: list[str], *, cwd: Path | None = None) -> str:
    return subprocess.run(
        argv, cwd=cwd, check=True, text=True, capture_output=True
    ).stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Stage 1A requires Python 3.11. Select the Colab 2025.07 runtime "
        "if it is still offered; do not continue under a mutable default runtime."
    )

smi = capture(["nvidia-smi"])
cuda_match = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi)
if cuda_match is None:
    raise RuntimeError("nvidia-smi did not report a CUDA compatibility version")
driver_cuda = tuple(int(part) for part in cuda_match.groups())
if driver_cuda < (12, 4):
    raise RuntimeError(
        "The planned cu124 wheel requires CUDA driver compatibility >=12.4; "
        f"got {driver_cuda}"
    )
gpu = capture(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ]
)
print(f"Python: {sys.version.split()[0]}")
print(f"GPU: {gpu}")
print(f"Driver CUDA compatibility: {driver_cuda[0]}.{driver_cuda[1]}")

In [ ]:
if REPOSITORY_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite existing checkout: {REPOSITORY_DIR}")
run(
    [
        "git",
        "clone",
        "--branch",
        PROJECT_REF,
        "--single-branch",
        REPOSITORY,
        str(REPOSITORY_DIR),
    ]
)
PROJECT_COMMIT = capture(["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR)
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Could not resolve the project checkout to an immutable commit")
if PROJECT_COMMIT != EXPECTED_PROJECT_COMMIT:
    raise RuntimeError(
        "The branch head does not match EXPECTED_PROJECT_COMMIT; refusing mutable code"
    )
if capture(["git", "status", "--porcelain"], cwd=REPOSITORY_DIR):
    raise RuntimeError("Fresh project checkout is unexpectedly dirty")
print(f"Resolved project branch to commit: {PROJECT_COMMIT}")

In [ ]:
PLANNED_REQUIREMENTS = (
    REPOSITORY_DIR / "environments/stage1a/requirements-colab-py311-cu124-planned.txt"
)
if not PLANNED_REQUIREMENTS.is_file():
    raise FileNotFoundError(
        f"Missing tracked Colab requirements plan: {PLANNED_REQUIREMENTS}"
    )
requirements_text = PLANNED_REQUIREMENTS.read_text(encoding="utf-8")
required_pins = (
    "torch==2.6.0",
    "transformer-lens==3.2.1",
    "transformers==4.57.3",
    "nnsight==0.6.1",
    "huggingface-hub==0.36.2",
    f"circuit-tracer.git@{UPSTREAM_COMMIT}",
)
missing_pins = [pin for pin in required_pins if pin not in requirements_text]
if missing_pins:
    raise RuntimeError(f"Tracked Colab plan is missing exact pins: {missing_pins}")

if VENV_DIR.exists():
    raise FileExistsError(f"Refusing to overwrite existing environment: {VENV_DIR}")
run([sys.executable, "-m", "venv", str(VENV_DIR)])
PYTHON = str(VENV_DIR / "bin/python")
run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
        "torch==2.6.0",
    ]
)
run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://pypi.org/simple",
        "--requirement",
        str(PLANNED_REQUIREMENTS),
    ]
)
run(
    [PYTHON, "-m", "pip", "install", "--no-deps", "--editable", "."],
    cwd=REPOSITORY_DIR,
)
run([PYTHON, "-m", "pip", "check"])

In [ ]:
verification_code = f'''
import importlib.metadata as metadata
import json
import torch

expected = {{
    "transformer-lens": "3.2.1",
    "transformers": "4.57.3",
    "nnsight": "0.6.1",
    "huggingface-hub": "0.36.2",
}}
for package, version in expected.items():
    actual = metadata.version(package)
    if actual != version:
        raise RuntimeError(f"{{package}}: expected {{version}}, got {{actual}}")
if metadata.version("torch").split("+", 1)[0] != "2.6.0":
    raise RuntimeError(f"Unexpected torch version: {{metadata.version('torch')}}")
if torch.version.cuda != "12.4" or not torch.cuda.is_available():
    raise RuntimeError(
        f"Expected an available cu124 runtime; torch CUDA={{torch.version.cuda}}, "
        f"available={{torch.cuda.is_available()}}"
    )
minimum_total_vram_bytes = 14 * 1024**3
minimum_free_vram_bytes = 12 * 1024**3
device_properties = torch.cuda.get_device_properties(0)
free_vram_bytes, total_vram_bytes = torch.cuda.mem_get_info(0)
cuda_readiness = {{
    "device_name": device_properties.name,
    "free_vram_bytes_before_load": free_vram_bytes,
    "minimum_free_vram_bytes": minimum_free_vram_bytes,
    "minimum_total_vram_bytes": minimum_total_vram_bytes,
    "official_batch_size": 256,
    "oom_may_still_occur": True,
    "policy_status": "unverified_preflight_floor",
    "passed": (
        total_vram_bytes >= minimum_total_vram_bytes
        and free_vram_bytes >= minimum_free_vram_bytes
    ),
    "total_vram_bytes": total_vram_bytes,
}}
print(json.dumps({{"cuda_readiness": cuda_readiness}}, sort_keys=True))
if not cuda_readiness["passed"]:
    raise RuntimeError(
        "Unverified preflight policy requires at least 14 GiB total and "
        "12 GiB free VRAM; this does not guarantee the run will avoid OOM"
    )
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The selected CUDA device does not support bfloat16")
bf16_operand = torch.ones((2, 2), device="cuda", dtype=torch.bfloat16)
bf16_result = bf16_operand @ bf16_operand
if not torch.isfinite(bf16_result).all().item():
    raise RuntimeError("CUDA bfloat16 execution probe returned non-finite values")
torch.cuda.synchronize()
distribution = metadata.distribution("circuit-tracer")
direct_url_text = distribution.read_text("direct_url.json")
if direct_url_text is None:
    raise RuntimeError("circuit-tracer direct_url.json is missing")
direct_url = json.loads(direct_url_text)
vcs = direct_url.get("vcs_info", {{}})
if direct_url.get("url") != "https://github.com/decoderesearch/circuit-tracer.git":
    raise RuntimeError(f"Unexpected circuit-tracer source: {{direct_url.get('url')}}")
if vcs.get("vcs") != "git" or vcs.get("commit_id") != "{UPSTREAM_COMMIT}":
    raise RuntimeError(f"Unexpected circuit-tracer VCS metadata: {{vcs}}")
print("Pinned environment and circuit-tracer provenance verified.")
'''
run([PYTHON, "-c", verification_code])

In [ ]:
try:
    from google.colab import userdata
except ImportError as exc:
    raise RuntimeError("This orchestration cell must run in Google Colab") from exc

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError(
        "Add an accessible Colab secret named HF_TOKEN before continuing"
    )

BASE_CONFIG = REPOSITORY_DIR / "configs/stage1a_gemma2_2b_official_reproduction.yaml"
if not BASE_CONFIG.is_file():
    raise FileNotFoundError(f"Missing tracked Stage 1A config: {BASE_CONFIG}")
config_text = BASE_CONFIG.read_text(encoding="utf-8")
for immutable_value in (MODEL_ID, MODEL_REVISION, TRANSCODER_ID, TRANSCODER_REVISION):
    if immutable_value not in config_text:
        raise RuntimeError(
            f"Tracked config does not contain immutable value: {immutable_value}"
        )

COLAB_CONFIG = Path("/content/stage1a_colab_resolved.yaml")
config_override_code = """
import pathlib
import sys
import yaml

source = pathlib.Path(sys.argv[1])
destination = pathlib.Path(sys.argv[2])
config = yaml.safe_load(source.read_text(encoding="utf-8"))
config["runtime"]["device"] = "cuda"
config["attribution"]["offload"] = "disk"
destination.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)
"""
run([PYTHON, "-c", config_override_code, str(BASE_CONFIG), str(COLAB_CONFIG)])

run_env = os.environ.copy()
run_env.update(
    {
        "HF_TOKEN": hf_token,
        "HUGGING_FACE_HUB_TOKEN": hf_token,
        "HF_HOME": "/content/cfsus-hf-cache",
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "TOKENIZERS_PARALLELISM": "false",
    }
)
run(
    [
        PYTHON,
        "scripts/stage1a/run_stage1a.py",
        "--config",
        str(COLAB_CONFIG),
        "--allow-download",
    ],
    cwd=REPOSITORY_DIR,
    env=run_env,
)
run(
    [
        PYTHON,
        "scripts/stage1a/validate_artifacts.py",
        "--artifact-dir",
        "results/stage1a",
        "--write-checksums",
    ],
    cwd=REPOSITORY_DIR,
    env=run_env,
)
run_env.pop("HF_TOKEN", None)
run_env.pop("HUGGING_FACE_HUB_TOKEN", None)
hf_token = None

In [ ]:
RESULTS_DIR = REPOSITORY_DIR / "results/stage1a"
small_result_names = (
    "environment_manifest.json",
    "asset_manifest.json",
    "colab_handoff_manifest.json",
    "attribution_summary.json",
    "intervention_summary.json",
    "semantics_summary.json",
    "checksums.sha256",
)
small_results = [RESULTS_DIR / name for name in small_result_names]
missing_results = [path.name for path in small_results if not path.is_file()]
if missing_results:
    raise RuntimeError(
        f"Validated run did not produce required summaries: {missing_results}"
    )
for path in small_results:
    if path.is_symlink():
        raise RuntimeError(f"Refusing to package symlink: {path.name}")
    if path.stat().st_size > 5 * 1024 * 1024:
        raise RuntimeError(f"Refusing to package result larger than 5 MiB: {path.name}")

artifact_records = {
    path.name: {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in small_results
}
runtime_manifest = {
    "artifacts": artifact_records,
    "assets": {
        "model": {"repository": MODEL_ID, "revision": MODEL_REVISION},
        "transcoder": {
            "repository": TRANSCODER_ID,
            "revision": TRANSCODER_REVISION,
        },
    },
    "project_commit": PROJECT_COMMIT,
    "schema_version": "1.0.0",
    "status": "validated_artifact_bundle",
    "upstream_commit": UPSTREAM_COMMIT,
}
RUNTIME_MANIFEST = Path("/content/stage1a_colab_run_manifest.json")
RUNTIME_MANIFEST.write_text(
    json.dumps(runtime_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

BUNDLE = Path("/content/stage1a-small-artifacts.zip")
with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [*small_results, RUNTIME_MANIFEST]:
        arcname = (
            f"results/stage1a/{path.name}" if path.parent == RESULTS_DIR else path.name
        )
        info = zipfile.ZipInfo(arcname, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        archive.writestr(info, path.read_bytes())

print(f"Small artifact bundle: {BUNDLE}")
print(f"Bundle SHA-256: {sha256_file(BUNDLE)}")
print(f"Runtime handoff manifest: {RUNTIME_MANIFEST}")